# Authentication

**Objective:** Verify credentials safely and issue short-lived access tokens.

## Simple version

In [ ]:
# Sign the user identity, then verify the signature before trusting it.
import jwt


secret = "demo-secret-with-at-least-32-characters"
token = jwt.encode({"sub": "user-1"}, secret, algorithm="HS256")
payload = jwt.decode(token, secret, algorithms=["HS256"])

print(payload["sub"])

## Polished version

Use Argon2 for passwords, move its CPU work off the event loop, and compare service API keys in constant time.

In [ ]:
# Authentication combines non-blocking password hashing and short-lived tokens.
import hmac
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone

import anyio
import jwt
from argon2 import PasswordHasher as Argon2Hasher
from argon2.exceptions import InvalidHashError, VerificationError


class PasswordHasher:
    def __init__(self) -> None:
        self._hasher = Argon2Hasher()

    async def hash(self, password: str) -> str:
        # Argon2 is intentionally expensive, so run it in a worker thread.
        return await anyio.to_thread.run_sync(self._hasher.hash, password)

    async def verify(self, password: str, encoded: str) -> bool:
        try:
            return await anyio.to_thread.run_sync(self._hasher.verify, encoded, password)
        except (VerificationError, InvalidHashError):
            return False


def verify_service_key(provided: str, expected: str) -> bool:
    # Constant-time comparison avoids leaking partial-key matches.
    return hmac.compare_digest(provided, expected)


@dataclass(frozen=True)
class User:
    id: int
    email: str
    password_hash: str


class AuthService:
    def __init__(self, secret: str) -> None:
        self.secret = secret
        self.passwords = PasswordHasher()
        self.users: dict[str, User] = {}

    async def register(self, email: str, password: str) -> User:
        email = email.strip().lower()
        if not 8 <= len(password) <= 128:
            raise ValueError("password must contain 8 to 128 characters")
        if email in self.users:
            raise ValueError("email already registered")
        user = User(len(self.users) + 1, email, await self.passwords.hash(password))
        self.users[email] = user
        return user

    async def login(self, email: str, password: str) -> str:
        user = self.users[email]
        if not await self.passwords.verify(password, user.password_hash):
            raise ValueError("invalid credentials")
        # Short expiry limits damage if an access token is stolen.
        expires = datetime.now(timezone.utc) + timedelta(minutes=15)
        return jwt.encode({"sub": str(user.id), "exp": expires}, self.secret, "HS256")


auth = AuthService("demo-secret-with-at-least-32-characters")
await auth.register("ada@example.com", "password123")
token = await auth.login("ada@example.com", "password123")
assert verify_service_key("service-key", "service-key")
print(token[:24] + "...")

## Applied in this repository

The REST [security adapter](../00P1-project-rest-api/app/infrastructure/security.py) offloads Argon2 and issues JWTs. The LLM [API-key dependency](../00P2-project-llm-api/app/interface/dependencies.py) uses constant-time comparison.